# Modeling Workflow
### Woojin Park, Korea University
### Copyright 2026, Korea University

This notebook preserves the original experimental flow while using repository-local data paths and a configurable class-wise sample size.


In [ ]:
from pathlib import Path
import os

# Change this value for larger experiments, for example 20000 or 40000.
SAMPLES_PER_CLASS = 1000
RANDOM_STATE = 42
FEATURE_EXTRACTION_SAMPLE_SIZE = 300

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CACHE_DIR = PROJECT_ROOT / '.cache'
(CACHE_DIR / 'matplotlib').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'numba').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'huggingface').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'huggingface' / 'datasets').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_DIR / 'matplotlib'))
os.environ.setdefault('NUMBA_CACHE_DIR', str(CACHE_DIR / 'numba'))
os.environ.setdefault('HF_HOME', str(CACHE_DIR / 'huggingface'))
os.environ.setdefault('HF_DATASETS_CACHE', str(CACHE_DIR / 'huggingface' / 'datasets'))
os.environ.setdefault('TRANSFORMERS_CACHE', str(CACHE_DIR / 'huggingface' / 'transformers'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')

INPUT_PATH = PROJECT_ROOT / 'data' / '02_preprocessing_outputs' / 'final_preprocessed_df.csv'
RUN_NAME = f'sample_{SAMPLES_PER_CLASS}_per_class'
MODELING_INPUT_DIR = PROJECT_ROOT / 'data' / '03_modeling_inputs' / RUN_NAME
MODELING_INPUT_DIR.mkdir(parents=True, exist_ok=True)

print('INPUT_PATH:', INPUT_PATH)
print('MODELING_INPUT_DIR:', MODELING_INPUT_DIR)


## Requirements

In [ ]:
# Run this only if dependencies are missing or if matplotlib/numpy import errors appear.
# After running with RUN_DEPENDENCY_INSTALL=True, restart the kernel and run again from the top.
RUN_DEPENDENCY_INSTALL = False

if RUN_DEPENDENCY_INSTALL:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--upgrade',
        '--force-reinstall',
        '-r',
        str(PROJECT_ROOT / 'requirements.txt'),
    ])
    print('Dependency install finished. Restart the kernel before continuing.')
else:
    print('Dependency install skipped. Set RUN_DEPENDENCY_INSTALL=True only when packages are missing.')


## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from umap import UMAP
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn import svm
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report)
from sklearn.metrics import RocCurveDisplay, auc
from sklearn.model_selection import (StratifiedKFold, cross_val_score, GridSearchCV)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


import datasets
from datasets import Dataset, DatasetDict
from datasets import load_dataset
from datasets import Features, Value, ClassLabel

from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer
from transformers import Trainer
from transformers import AutoModel
from transformers import pipeline

from torch.nn.functional import cross_entropy


## UDF

In [ ]:
def label_int2str(row):
    return dataset["train"].features["label"].int2str(row)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

def plot_confusion_matrix(y_preds, y_true, labels):
    label_ids = list(range(len(labels)))
    cm = confusion_matrix(y_true, y_preds, labels=label_ids, normalize="true")
    cm = np.nan_to_num(cm, nan=0.0)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    plt.title("Normalized confusion matrix")
    plt.show()


def forward_pass_with_label(batch):
    # Place all input tensors on the same device as the model
    inputs = {k:v.to(device) for k,v in batch.items()
              if k in tokenizer.model_input_names}

    with torch.no_grad():
        output = model(**inputs)
        pred_label = torch.argmax(output.logits, axis=-1)
        loss = cross_entropy(output.logits, batch["label"].to(device),
                             reduction="none")

    # Place outputs on CPU for compatibility with other dataset columns
    return {"loss": loss.cpu().numpy(),
            "predicted_label": pred_label.cpu().numpy()}

def analyze_error_test_set(df_encoded_src, split='validation'):
    # Stable error analysis based on Trainer predictions.
    # This avoids device mismatch issues on CPU/MPS/CUDA environments.
    preds_output = trainer.predict(df_encoded_src[split])
    logits = torch.tensor(preds_output.predictions)
    labels = torch.tensor(preds_output.label_ids)
    losses = cross_entropy(logits, labels, reduction='none').numpy()
    predicted_labels = np.argmax(preds_output.predictions, axis=-1)

    df_encoded_src.set_format('pandas')
    df_test = df_encoded_src[split][:][['text', 'label']].copy()
    df_test['predicted_label'] = predicted_labels
    df_test['loss'] = losses
    df_test['label'] = df_test['label'].apply(lambda x: label_int2str.get(x))
    df_test['predicted_label'] = df_test['predicted_label'].apply(lambda x: label_int2str.get(x))
    return df_test

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

def encode_dataset(df_source):
  df_encoded = df_source.map(tokenize, batched=True, batch_size=None)
  df_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
  return df_encoded

def extract_hidden_states(batch):
    # Place model inputs on the GPU
    inputs = {k:v.to(device) for k,v in batch.items()
              if k in tokenizer.model_input_names}
    # Extract last hidden states
    with torch.no_grad():
        last_hidden_state = model(**inputs).last_hidden_state
    # Return vector for [CLS] token
    return {"hidden_state": last_hidden_state[:,0].cpu().numpy()}

def extract_hidden_states_df(df_source):
  df_encoded = encode_dataset(df_source)
  df_hidden = df_encoded.map(extract_hidden_states, batched=True)
#   torch.cuda.empty_cache()
  return df_hidden


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision = precision_score(labels, preds, average="weighted", zero_division=0)
    recall = recall_score(labels, preds, average="weighted", zero_division=0)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


def plot_confusion_matrix(y_preds, y_true, labels):
    label_ids = list(range(len(labels)))
    cm = confusion_matrix(y_true, y_preds, labels=label_ids, normalize="true")
    cm = np.nan_to_num(cm, nan=0.0)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    plt.title("Normalized confusion matrix")
    plt.show()

def measure_model_performance(trainer, df_input, y_labels, labels_list):
    preds_output = trainer.predict(df_input)
    print(preds_output.metrics)
    y_preds = np.argmax(preds_output.predictions, axis=1)
    y_true = np.array(y_labels)
    correct_count = int((y_preds == y_true).sum())
    total_count = int(len(y_true))
    incorrect_count = total_count - correct_count
    accuracy_pct = 100 * correct_count / total_count if total_count else 0
    print(f'Correct predictions: {correct_count} / {total_count} ({accuracy_pct:.2f}%)')
    print(f'Incorrect predictions: {incorrect_count} / {total_count}')
    plot_confusion_matrix(y_preds, y_true, labels_list)

def train_model(model_ckpt, training_args, df_encoded_src, num_labels):
    # Create Model
    model = (AutoModelForSequenceClassification
            .from_pretrained(model_ckpt, num_labels=num_labels)
            .to(device))
    model_name = f"models/{model_ckpt}-finetuned-turkish-tweets"
    training_args.output_dir = model_name
    trainer = Trainer(model=model, args=training_args,
                    compute_metrics=compute_metrics,
                    train_dataset=df_encoded_src["train"],
                    eval_dataset=df_encoded_src["validation"],
                    tokenizer=tokenizer)
    trainer.train()
    return trainer

# Load Dataset

In [ ]:
#### Data Preparation step.
#### input the final_preprocessed_df.csv file path here
df = pd.read_csv(INPUT_PATH, low_memory=False)
df.head()

In [ ]:
df.info()

In [ ]:
df.head(3)

## Data Pre-processing

In [ ]:
#### Pre-processing Step 0. Build dataframe for modeling : select only necessary features
df_pre = df[['title_with_selftext','class_group']]
df_pre.columns = ['text','label_str']

In [ ]:
df_pre.head(3)

In [ ]:
#### Pre-processing Step 1. Check the class distribution
df_pre.label_str.value_counts()

In [ ]:
#### Pre-processing Step 2. Check the missing values
print(df_pre['label_str'].isnull().sum())
df_pre[df_pre['label_str'].isnull()].transpose()

In [ ]:
#### Pre-processing Step 2-1. Missing value treatmnet
df_pre = df_pre.dropna(subset='label_str')

In [ ]:
#### Pre-processing Step 3-1.Generate label feature for modeling
df_pre['label'] = df_pre.label_str.map({
    'Depression_Group': 0,
    'Neutral_Group': 1,
    'Happy_Group': 2,
})

#### Pre-processing Step 4. Select relevant features
df_pre = df_pre[['text','label']]
df_pre.head(3)

## Modeling  

In [ ]:
#### Modeling Step 1. Modeling Data Preparation
###### Generate class dataset
df_depression = df_pre[df_pre.label==0]
df_neutral = df_pre[df_pre.label==1]
df_happy = df_pre[df_pre.label==2]

###### Generate randomly sampled class dataset
df_depression_sampled = resample(df_depression, replace=True, n_samples=SAMPLES_PER_CLASS, random_state=RANDOM_STATE) # reproducible results
df_neutral_sampled = resample(df_neutral, replace=True, n_samples=SAMPLES_PER_CLASS, random_state=RANDOM_STATE) # reproducible results
df_happy_sampled = resample(df_happy, replace=True, n_samples=SAMPLES_PER_CLASS, random_state=RANDOM_STATE) # reproducible results

###### Concat sampled dataset
df_sampled = pd.concat([df_depression_sampled, df_neutral_sampled, df_happy_sampled])
###### Check the sample distribution for classes
df_sampled.label.value_counts()

In [ ]:
#### Modeling Step 2. Dataset Split (train / validation / test) : 75% / 15% / 10 % respectively
df_train_dataset, df_validation_dataset, df_test_dataset = \
  np.split(df_sampled.sample(frac=1, random_state=RANDOM_STATE), [int(.75*len(df_sampled)), int(.9*len(df_sampled))])

###### Check the train/ validation / test count after the split
print(f'train data size : {len(df_train_dataset)}')
print(f'validation data size : {len(df_validation_dataset)}')
print(f'test data size : {len(df_test_dataset)}')

###### Save it as csv for load
df_train_dataset.to_csv(MODELING_INPUT_DIR / 'train_dataset.csv', index=False)
df_validation_dataset.to_csv(MODELING_INPUT_DIR / 'validation_dataset.csv', index=False)
df_test_dataset.to_csv(MODELING_INPUT_DIR / 'test_dataset.csv', index=False)


###### Formatted data as 'DatasetDict' type for the further modeling step
class_names = ["Depression_Group", "Neutral_Group", "Happy_Group"]
sentiment_features = Features({'text': Value('string'), 'label': ClassLabel(names=class_names)}) ### feature maping from integer to class
file_dict = {'train': str(MODELING_INPUT_DIR / 'train_dataset.csv'),
             'validation': str(MODELING_INPUT_DIR / 'validation_dataset.csv'),
             'test': str(MODELING_INPUT_DIR / 'test_dataset.csv')}
dataset = load_dataset('csv', data_files=file_dict, delimiter=",", column_names=['text', 'label'], features=sentiment_features, skiprows=1)

In [ ]:
### DatasetDict format checking
dataset.set_format(type="pandas")
dataset

In [ ]:
#### Modeling Step 3.label maping Convert Label From Integer to String
###### check the label name and number
print(set(dataset['train']['label']))
print(dataset['train'].features['label'])


###### Build the train/ validation / test dataset with label mapping
df_train = dataset["train"][:]
df_test = dataset["test"][:]
df_validation = dataset["validation"][:]

df_train["label_str"] = df_train["label"].apply(label_int2str)
df_test["label_str"] = df_test["label"].apply(label_int2str)
df_validation["label_str"] = df_validation["label"].apply(label_int2str)

print(df_train.shape)
print(df_validation.shape)
print(df_test.shape)

In [ ]:
df_train.head(3)

In [ ]:
#### Modeling Step 4. Class distribution Inspection

###### Class Distributions in Training Set
df_train["label_str"].value_counts(ascending=True).plot.barh()
plt.title("Class Distributions in Training Set")
plt.show()

###### Class Distributions in Validation Set
plt.title("Class Distributions in Validation Set")
df_validation["label_str"].value_counts(ascending=True).plot.barh()
plt.show()

###### Class Distributions in Test Set
plt.title("Class Distributions in Test Set")
df_test["label_str"].value_counts(ascending=True).plot.barh()
plt.show()

In [ ]:
#### Modeling Step 5. Subreddit sample length distribution

df_train["Words Per Subreddit Post"] = df_train["text"].str.split().apply(len)
df_train.boxplot("Words Per Subreddit Post", by="label", grid=False,
 showfliers=False, color="black")
plt.suptitle("Training Set")
plt.xlabel("")
plt.show()
df_train.drop(columns=['Words Per Subreddit Post'],inplace=True)

df_validation["Words Per Subreddit Post"] = df_validation["text"].str.split().apply(len)
df_validation.boxplot("Words Per Subreddit Post", by="label", grid=False,
 showfliers=False, color="black")
plt.suptitle("Validation Set")
plt.xlabel("")
plt.show()
df_validation.drop(columns=['Words Per Subreddit Post'],inplace=True)

df_test["Words Per Subreddit Post"] = df_test["text"].str.split().apply(len)
df_test.boxplot("Words Per Subreddit Post", by="label", grid=False,
 showfliers=False, color="black")
plt.suptitle("Test Set")
plt.xlabel("")
plt.show()
df_test.drop(columns=['Words Per Subreddit Post'],inplace=True)

In [ ]:
#### Modeling Step 6. Modeling loading & Tokenization
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

##### example case output after tokenization
encoded_text = tokenizer("I am so depressed")
print("encoded_text: ",encoded_text)
tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
print(encoded_text,len(encoded_text.input_ids))
print(tokens)

print(tokenizer.convert_tokens_to_string(tokens))

print("tokenizer.vocab_size:",tokenizer.vocab_size)
print("Model max context length:",tokenizer.model_max_length)


###### case output
text_raw = df_train['text'][2]
tokenized_text = tokenizer(text_raw) # raw text converted to index numbers
tokenized_text_tokens = tokenizer.convert_ids_to_tokens(tokenized_text.input_ids)
print("tokenized_text - indices: ",tokenized_text['input_ids'])
print("tokenized_text - attention mask: ",tokenized_text['attention_mask'])
print("tokenized_text_tokens: ",tokenized_text_tokens)

In [ ]:
#### example output

In [ ]:
df_input_ids = pd.DataFrame(tokenizer(list(df_train['text'][:3]))['input_ids'])
df_input_ids

In [ ]:
df_attention_mask = pd.DataFrame(tokenizer(list(df_train['text'][:3]))['attention_mask'])
df_attention_mask

In [ ]:
###### train / test tokenization
df_train = Dataset.from_pandas(df_train)
df_test = Dataset.from_pandas(df_test)
df_encoded_train = df_train.map(tokenize, batched=True, batch_size=None)
df_encoded_test = df_test.map(tokenize, batched=True, batch_size=None)

tokens2ids = list(zip(tokenizer.all_special_tokens, tokenizer.all_special_ids))
tokens2ids

In [ ]:
###### label encoding
le = LabelEncoder()
le.fit(dataset['train'].features['label'].names)
##### label int to string mapping
labels_int_list = list(np.unique(dataset['train']['label']))
label_int2str = dict.fromkeys(labels_int_list)
label_int2str = {k:le.inverse_transform([k])[0] for k,v in label_int2str.items()}
label_int2str

In [ ]:
#### Modeling Step 7. Load pretrained DistilBERT-Base-Uncased Model as Feature Extractor

model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: ",device)
model = AutoModel.from_pretrained(model_ckpt).to(device)

In [ ]:
#### Modeling Step 8. Extracting the last hidden states
np.object = object

# Feature extraction is for visualization/error inspection and can be slow on CPU.
# Keep the original workflow, but use a smaller subset by default for quick testing.
dataset.reset_format()
hidden_dataset = DatasetDict({
    split: dataset[split].select(range(min(FEATURE_EXTRACTION_SAMPLE_SIZE, len(dataset[split]))))
    for split in dataset.keys()
})

df_hidden = extract_hidden_states_df(hidden_dataset)
df_hidden

In [ ]:
###### Apply hidden_state information
X_train = np.array(df_hidden["train"]["hidden_state"])
X_valid = np.array(df_hidden["validation"]["hidden_state"])
X_test = np.array(df_hidden["test"]["hidden_state"])

y_train = np.array(df_hidden["train"]["label"])
y_valid = np.array(df_hidden["validation"]["label"])
y_test = np.array(df_hidden["test"]["label"])

X_train.shape, y_train.shape

### Training Methodology: Fine Tuning BERT

In [ ]:
#### Modeling step 9. Training Methodology - Fine Tuning BERT Create dataset
df_encoded = encode_dataset(dataset)
df_encoded

# Hyperparameter tuning (wandb)

In [ ]:
# Optional: run only if W&B is missing in your environment.
# %pip install wandb

In [ ]:
# Optional W&B login for hyperparameter sweep.
# import wandb
# wandb.login()

In [ ]:
# Optional W&B sweep template from the original experiment.
# Run this section only when you intentionally want hyperparameter search.
#
# sweep_configuration = {
#     'method': 'bayes',
#     'name': 'sweep',
#     'metric': {'goal': 'minimize', 'name': 'eval_loss'},
#     'parameters': {
#         'batch_size': {'values': [16, 32]},
#         'epochs': {'values': [2, 3, 5]},
#         'weight_decay': {'values': [1e-2, 1e-3, 1e-4]},
#         'lr': {'max': 5e-5, 'min': 1e-5},
#      }
# }
# sweep_id = wandb.sweep(sweep=sweep_configuration, project='distilbert-sample-classification')

In [ ]:
# Optional: retrieve best parameters from a completed W&B sweep.
# Replace the path with your own entity/project/sweep id.
#
# import wandb
# api = wandb.Api()
# sweep = api.sweep('entity/project-name/sweeps/sweep-id')
# best_run = sweep.best_run(order='eval_loss')
# best_parameters = best_run.config
# print(best_parameters)

In [ ]:
#### Modeling step 10. Set default fine-tuning parameters
import inspect

best_parameters = globals().get('best_parameters', {
    'batch_size': 16,
    'epochs': 2,
    'lr': 2e-5,
    'weight_decay': 1e-3,
})

batch_size = best_parameters['batch_size']
logging_steps = max(1, len(df_encoded['train']) // batch_size)
num_train_epochs = best_parameters['epochs']
lr_initial = best_parameters['lr']
weight_decay = best_parameters.get('weight_decay', 1e-3)
output_dir = PROJECT_ROOT / 'models' / f'distilbert_{RUN_NAME}'

training_arg_kwargs = dict(
    output_dir=str(output_dir),
    num_train_epochs=num_train_epochs,
    learning_rate=lr_initial,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=weight_decay,
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    push_to_hub=False,
    report_to='none',
    log_level='error',
)

if 'eval_strategy' in inspect.signature(TrainingArguments).parameters:
    training_arg_kwargs['eval_strategy'] = 'epoch'
else:
    training_arg_kwargs['evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**training_arg_kwargs)
training_args

In [ ]:
#### Modeling step 11. Fine-Tuning for 'distilbert-base-uncased' model
model_ckpt = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=dataset['train'].features['label'].num_classes,
).to(device)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=df_encoded['train'],
    eval_dataset=df_encoded['validation'],
    compute_metrics=compute_metrics,
)
trainer_signature = inspect.signature(Trainer).parameters
if 'processing_class' in trainer_signature:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_signature:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)
trainer.train()

In [ ]:

#### Modeling step 12. Model evaluation Confuxion matrix
labels_names_list = dataset['train'].features['label'].names

###### validation set
measure_model_performance(trainer, df_encoded["validation"], df_encoded["validation"]['label'], labels_names_list)

In [ ]:
###### test set
measure_model_performance(trainer, df_encoded["test"], df_encoded["test"]['label'], labels_names_list)

In [ ]:
###### Error Analysis on Test Set
model = trainer.model
df_test = analyze_error_test_set(df_encoded_src=df_encoded)
df_test.sort_values("loss", ascending=False).head(10)

In [ ]:
###### Loss Case inspection

In [ ]:
df_test.sort_values("loss", ascending=False).iloc[0]['text']

In [ ]:
df_test.sort_values("loss", ascending=False).iloc[0][['label','predicted_label']]

In [ ]:
df_test.sort_values("loss", ascending=False).iloc[1]['text']

In [ ]:
df_test.sort_values("loss", ascending=False).iloc[1][['label','predicted_label']]

In [ ]:
#### Modeling step 13. Model save step
MODEL_OUTPUT_DIR = PROJECT_ROOT / 'models' / f'distilbert_{RUN_NAME}'
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))
print('Saved model to:', MODEL_OUTPUT_DIR)

In [ ]:
###### Use saved model and load pipeline functionality
classifier = pipeline('text-classification', model=str(MODEL_OUTPUT_DIR), tokenizer=str(MODEL_OUTPUT_DIR))

###### evaluation performance by unit test
custom_tweet = 'I saw a movie today and it was really good.'
preds = classifier(custom_tweet, return_all_scores=True)
print(preds)

In [ ]:
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.title(f'"{custom_tweet}"')
plt.ylabel("Class probability (%)")
plt.show()

## Model performance evaluation demo

In [ ]:
#### depression class

In [ ]:
custom_tweet = \
 "During school days, there are moments when I feel a sense of happiness, perhaps when I receive praise from a teacher or when I achieve a personal goal. \
However, underlying that happiness, there's often a deeper feeling of despair, like when I struggle to keep up with my classmates or when I feel isolated in a crowd. \
Even though there are moments of joy, the weight of hopelessness seems to overshadow them, leaving me feeling more alone than ever."



preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)

In [ ]:
custom_tweet = \
"At work, there are instances when I experience moments of contentment, such as when I finish a project ahead of schedule or receive positive feedback from a colleague.\
Yet, despite these brief moments of happiness, there's a persistent sense of emptiness that lingers, stemming from feelings of insecurity or dissatisfaction with my career. \
Despite my efforts to focus on the positives, the feeling of despair remains, making it difficult to fully enjoy the successes I achieve."


preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)

In [ ]:
custom_tweet = \
"Within my family, there are times when I share moments of laughter and love with my loved ones, cherishing the bonds we share.\
However, amidst these moments of happiness, there's an underlying feeling of sorrow that I can't seem to shake off, stemming from unresolved conflicts or distant relationships. \
Despite the warmth of family connections, the weight of melancholy prevails, casting a shadow over our interactions and leaving me yearning for deeper connection and understanding."


preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)

In [ ]:
### happy class

In [ ]:

custom_tweet = \
"During school days, there are moments when I feel down, maybe because of a bad grade or a disagreement with a friend. \
But amidst those gloomy times, there are also moments of happiness, like when I solve a problem I've been struggling with\
or when I share a laugh with my classmates. Even though the sad moments weigh on me, the happiness I feel is a bit stronger.\
It's like finding a small flower blooming in the midst of a barren field, a tiny glimmer of light in the darkness. \
Despite the challenges, those moments of joy keep me going, reminding me that there's always something to smile about,\
even in the toughest of times."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
preds_df.transpose()

In [ ]:

custom_tweet = \
 "During family gatherings, there are occasions when tensions run high, perhaps due to disagreements or unresolved conflicts. \
But amid the occasional conflicts, there are also moments of happiness, like when we share stories and laughter around the dinner table or reminisce about fond memories. \
Even though arguments may arise, the warmth of family bonds prevails. It's like finding harmony in the midst of discord, a glimmer of love amidst the strife. \
Despite the occasional disagreements, those moments of connection keep us united, reminding us of the strength of family ties."



preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)



In [ ]:
custom_tweet = \
"At work, there are days when I feel overwhelmed, especially when deadlines are looming or when I face criticism from my boss. \
But in the midst of those stressful times, there are also moments of happiness, like when I successfully complete a project or receive recognition for my hard work. \
Even though the pressure gets to me, the joy of accomplishment outweighs the stress. It's like finding a silver lining in a stormy sky, a brief moment of relief amidst the chaos. \
Despite the challenges, those small victories keep me motivated, reminding me that perseverance pays off."


preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)




In [ ]:
#### neutral class

In [ ]:

custom_tweet = \
"Recent studies suggest that the new policy implemented by the government has elicited a mixed response from experts in the field.\
 While some argue for its potential benefits in addressing socioeconomic disparities, others remain skeptical, citing potential unintended consequences.\
 Despite the divergent opinions, further research is needed to assess the long-term implications of this policy shift.."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)



In [ ]:

custom_tweet = \
"In the latest quarterly report, the company announced steady growth in revenue and market share,\
signaling stability in the face of economic uncertainty.However, analysts caution against complacency,\
highlighting potential challenges in the competitive landscape and evolving consumer preferences. \
Despite the cautious outlook, the company remains optimistic about its prospects for future growth."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
preds_df.transpose()

In [ ]:

custom_tweet = \
"The findings from the survey reveal a nuanced perspective on the issue, with respondents expressing a range of viewpoints.\
 While some respondents voiced support for the proposed initiative, others expressed reservations, citing concerns about feasibility and implementation challenges.\
  Despite the differing opinions, there is consensus on the need for further dialogue and collaboration to address the complex issues at hand."


preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)



In [ ]:

custom_tweet = \
"Even on a lively school day, amidst the laughter and chatter of friends,\
 there's a quiet ache in my heart.It's like being part of the joy while \
 carrying a hidden weight,a faint feeling of emptiness that mutes the excitement,\
 reminding me that even in the midst of happiness, there's a subtle shadow."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)

In [ ]:
custom_tweet = \
"Even though the halls are filled with laughter and chatter, there's this feeling weighing on me.\
It's like riding the wave of excitement while also carrying a bit of sadness, a reminder that life isn't always sunshine and rainbows.\
Despite the energy around me, there's a cloud in my mind, making everything seem a bit less bright.\
It's a mix of happy and sad, like a seesaw where neither side is quite up or down. \
But I still hold onto hope, believing things will get better, even if they're a bit uncertain right now."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
preds_df.transpose()
